# 01. Exploracion inicial de reflejos endoscopicos

Este notebook esta pensado como primer contacto con el proyecto.

Objetivos:
- aprender a cargar imagenes endoscopicas desde `data/raw/`
- revisar hasta 10 ejemplos
- inspeccionar tamano, brillo y saturacion
- producir una mascara candidata de reflejos especulares con un baseline simple
- entender que entradas y salidas necesita el proyecto segun la tarea


## Que poner en `data/raw/`

Coloca aqui imagenes como `.png`, `.jpg`, `.jpeg`, `.bmp`, `.tif` o `.tiff`.

Ejemplo de estructura:

```text
data/raw/
  paciente_001_frame_001.png
  paciente_001_frame_002.png
  caso_a_01.jpg
```

No importa si al inicio no tienes etiquetas. Primero vamos a aprender a consumir las imagenes.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from laluzqueopaca.models.baseline import detect_specular_highlights_bgr

plt.style.use("default")
pd.set_option("display.max_columns", 20)


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

PROJECT_ROOT, RAW_DIR

In [ ]:
def list_image_paths(directory: Path, limit: int = 10):
    image_paths = [
        path for path in sorted(directory.rglob("*"))
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]
    return image_paths[:limit]


def load_bgr_image(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"No se pudo leer la imagen: {path}")
    return image


def bgr_to_rgb(image: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def compute_image_summary(image_bgr: np.ndarray) -> dict:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    saturation = hsv[:, :, 1]
    value = hsv[:, :, 2]
    mask = detect_specular_highlights_bgr(image_bgr)

    return {
        "height": int(image_bgr.shape[0]),
        "width": int(image_bgr.shape[1]),
        "mean_gray": float(gray.mean()),
        "max_gray": int(gray.max()),
        "mean_saturation": float(saturation.mean()),
        "mean_value": float(value.mean()),
        "candidate_reflection_ratio": float((mask > 0).mean()),
    }


def show_image_triplet(path: Path):
    image_bgr = load_bgr_image(path)
    image_rgb = bgr_to_rgb(image_bgr)
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    value = hsv[:, :, 2]
    mask = detect_specular_highlights_bgr(image_bgr)

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(image_rgb)
    axes[0].set_title("Imagen original")
    axes[1].imshow(value, cmap="gray")
    axes[1].set_title("Canal V (brillo)")
    axes[2].imshow(mask, cmap="gray")
    axes[2].set_title("Mascara candidata")
    axes[3].imshow(image_rgb)
    axes[3].imshow(mask, cmap="autumn", alpha=0.35)
    axes[3].set_title("Overlay")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(path.name, fontsize=14)
    plt.tight_layout()
    plt.show()


## Paso 1. Encontrar hasta 10 imagenes

In [ ]:
image_paths = list_image_paths(RAW_DIR, limit=10)
len(image_paths), image_paths[:3]

In [ ]:
if not image_paths:
    print("No se encontraron imagenes en data/raw.")
    print("Siguiente accion sugerida: coloca entre 5 y 10 imagenes endoscopicas y vuelve a correr el notebook.")
else:
    print("Imagenes detectadas:")
    for path in image_paths:
        print(" -", path.name)


## Paso 2. Resumen tabular de las primeras 10 imagenes

In [ ]:
rows = []

for path in image_paths:
    image_bgr = load_bgr_image(path)
    summary = compute_image_summary(image_bgr)
    rows.append({"filename": path.name, **summary})

summary_df = pd.DataFrame(rows)
summary_df

Si `candidate_reflection_ratio` sale muy alto o muy bajo, no pasa nada. Eso solo te indica que el baseline inicial necesita ajuste.

## Paso 3. Visualizar una imagen a detalle

In [ ]:
if image_paths:
    show_image_triplet(image_paths[0])
else:
    print("Aun no hay imagenes para visualizar.")


## Paso 4. Ver varias imagenes seguidas

In [ ]:
for path in image_paths[:5]:
    show_image_triplet(path)


## Que entradas y salidas podria necesitar el proyecto

### Opcion A. Deteccion o segmentacion de reflejos
- Entrada: imagen endoscopica
- Salida: mascara binaria donde 1 indica reflejo especular

### Opcion B. Supresion o restauracion de reflejos
- Entrada: imagen endoscopica, y a veces mascara de reflejo
- Salida: imagen corregida sin regiones saturadas visibles

### Opcion C. Pipeline combinado
- Entrada: imagen endoscopica
- Salida 1: mascara de reflejos
- Salida 2: imagen restaurada
- Salida 3: metricas o puntajes de calidad


## Que deberias guardar desde el inicio

Aunque al principio no tengas etiquetas, intenta construir una tabla de control como esta:

- `image_id`
- `filename`
- `modality` si la conoces, por ejemplo WLI o NBI
- `height` y `width`
- `has_visible_reflection` como revision manual simple
- `mask_path` cuando ya tengas mascaras
- `notes` para observaciones


In [ ]:
dataset_register = summary_df.copy() if not summary_df.empty else pd.DataFrame(columns=[
    "image_id",
    "filename",
    "modality",
    "height",
    "width",
    "has_visible_reflection",
    "mask_path",
    "notes",
])

if not dataset_register.empty:
    dataset_register = dataset_register.rename(columns={"filename": "image_id"})
    dataset_register["filename"] = dataset_register["image_id"]
    dataset_register["modality"] = "unknown"
    dataset_register["has_visible_reflection"] = np.where(
        dataset_register["candidate_reflection_ratio"] > 0.001, "maybe", "review"
    )
    dataset_register["mask_path"] = ""
    dataset_register["notes"] = ""
    dataset_register = dataset_register[[
        "image_id",
        "filename",
        "modality",
        "height",
        "width",
        "has_visible_reflection",
        "mask_path",
        "notes",
    ]]

dataset_register

## Siguientes pasos sugeridos

1. Conseguir entre 10 y 30 imagenes reales y colocarlas en `data/raw/`.
2. Volver a correr este notebook y revisar visualmente las mascaras candidatas.
3. Elegir 5 ejemplos y anotar manualmente si el reflejo fue detectado bien o mal.
4. Ajustar el baseline en `src/laluzqueopaca/models/baseline.py`.
5. Cuando ya tengas criterio visual, decidir si el proyecto empezara por deteccion o por supresion.
